# 8) Demand response — Exercises

**Goals:** account for the value of consumption given up and for the scarcity rents generators earn, see how the scarcity price follows consumers' willingness to pay, push the system into involuntary shedding, and compare load shifting with pumped storage of the same size.

The setup cell holds the lecture's weeks and its model as a function `week(stress, dr, shift, load_scale)`, plus `pypsa_week` for checking. Power in MW, energy in MWh, prices in €/MWh, money in € per week.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [ ]:
import logging
import warnings

import linopy
import numpy as np
import pandas as pd
import pypsa
import xarray as xr

logging.getLogger("pypsa").setLevel(logging.ERROR)
logging.getLogger("linopy").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)

# ---- the lecture's weeks and model (same seed, same numbers) ----
T = 168
h = np.arange(T)
hour, day = h % 24, h // 24
rng = np.random.default_rng(3)
LOAD = (2000 + 300 * np.sin(np.pi * (hour - 6) / 12).clip(0) + 750 * np.exp(-((hour - 19) / 2.0) ** 2)
        - 150 * (day >= 5) + rng.normal(0, 30, T))
cloud = np.repeat([0.9, 0.6, 1.0, 0.8, 0.4, 1.0, 0.9], 24)
SOLAR = 2500 * 0.75 * np.clip(np.sin(np.pi * (hour - 6) / 13), 0, None) * cloud
WIND = 1000 * np.clip(0.3 + 0.25 * np.sin(2 * np.pi * h / 96), 0.02, 0.9)
DR = pd.DataFrame({"p_nom": [300, 250], "wtp": [300.0, 1000.0]}, index=pd.Index(["industry", "commercial"], name="dr"))
VOLL = 3000.0


def system(stress=False):
    thermal = pd.DataFrame({"p_nom": [350, 900, 600, 800, 400], "mc": [12.0, 89.1, 94.2, 104.1, 128.9]},
                           index=pd.Index(["nuclear", "ccgt", "coal", "imports", "ocgt"], name="gen"))
    free, inflow = SOLAR + WIND + 300.0, 120.0
    if stress:
        free, inflow = 0.5 * SOLAR + 0.2 * WIND + 300.0, 40.0
        thermal.loc[["imports", "coal"], "p_nom"] = 0
    return thermal, free, inflow


def week(stress=False, dr=DR, shift=None, load_scale=1.0):
    """The lecture's model. Returns a dict with the cost (EUR, incl. the value of lost consumption), prices
    (EUR/MWh), generation (DataFrame), demand reduction per group (DataFrame), shedding (MW) and the model."""
    thermal, free, inflow = system(stress)
    load = LOAD * load_scale
    m = linopy.Model()
    t, G, D = pd.RangeIndex(T, name="t"), thermal.index, dr.index
    g = m.add_variables(lower=0, upper=xr.DataArray(np.tile(thermal.p_nom.to_numpy(), (T, 1)), coords=[t, G]), name="g")
    f = m.add_variables(lower=0, upper=xr.DataArray(free, coords=[t]), name="free")
    d = m.add_variables(lower=0, upper=1000, coords=[t], name="turbine")
    s = m.add_variables(lower=0, coords=[t], name="spill")
    e = m.add_variables(lower=0, upper=30_000, coords=[t], name="level")
    m.add_constraints(e - e.roll(t=1) + d + s == inflow, name="water")
    r = m.add_variables(lower=0, upper=xr.DataArray(np.tile(dr.p_nom.to_numpy(), (T, 1)), coords=[t, D]), name="reduce")
    shed = m.add_variables(lower=0, coords=[t], name="shed")
    supply = g.sum("gen") + f + d + r.sum("dr") + shed
    cost = ((g * xr.DataArray(thermal.mc.to_numpy(), coords=[G])).sum()
            + (r * xr.DataArray(dr.wtp.to_numpy(), coords=[D])).sum() + VOLL * shed.sum())
    if shift:
        later = m.add_variables(lower=0, upper=shift["p"], coords=[t], name="delay")
        back = m.add_variables(lower=0, upper=shift["p"], coords=[t], name="catch_up")
        backlog = m.add_variables(lower=0, upper=shift["e"], coords=[t], name="backlog")
        m.add_constraints(backlog - backlog.roll(t=1) - later + back == 0, name="backlog_balance")
        supply = supply + later - back
    m.add_constraints(supply == xr.DataArray(load, coords=[t]), name="balance")
    m.add_objective(cost)
    m.solve(solver_name="highs", output_flag=False)
    return {"cost": m.objective.value, "price": m.constraints["balance"].dual.to_numpy() + 0.0,
            "gen": g.solution.to_pandas(), "reduce": r.solution.to_pandas(), "shed": shed.solution.to_numpy(), "model": m}


def pypsa_week(stress=False, dr=DR, load_scale=1.0):
    """The same week in PyPSA, for checking."""
    thermal, free, inflow = system(stress)
    n = pypsa.Network()
    n.set_snapshots(h)
    n.add("Bus", "HR")
    n.add("Load", "load", bus="HR", p_set=LOAD * load_scale)
    n.add("Generator", "free", bus="HR", p_nom=free.max(), p_max_pu=free / free.max(), marginal_cost=0)
    for gen, r in thermal.iterrows():
        if r.p_nom > 0:
            n.add("Generator", gen, bus="HR", p_nom=r.p_nom, marginal_cost=r.mc)
    n.add("StorageUnit", "reservoir", bus="HR", p_nom=1000, max_hours=30, p_min_pu=0.0, inflow=inflow,
          cyclic_state_of_charge=True)
    for name, r in dr.iterrows():
        n.add("Generator", f"dr_{name}", bus="HR", p_nom=r.p_nom, marginal_cost=r.wtp)
    n.add("Generator", "shedding", bus="HR", p_nom=(LOAD * load_scale).max(), marginal_cost=VOLL)
    n.optimize(solver_name="highs", log_to_console=False, progress=False)
    return n

### Warm-ups

**1. What did the consumers give up?**

Write `lost_value(result, dr=DR)` returning, per consumer group, the energy not consumed (MWh) and its value at the group's willingness to pay (€), plus a row for involuntary shedding valued at VOLL. Check the lecture's stressed week: industry gave up 6,933 MWh.

In [ ]:
def lost_value(result: dict, dr: pd.DataFrame = DR) -> pd.DataFrame:
    """Energy not consumed (MWh) and its value (EUR) per consumer group and for involuntary shedding."""
    ...  # YOUR CODE HERE

In [ ]:
stressed = week(stress=True)
lost = lost_value(stressed)
assert np.isclose(lost.loc["industry", "MWh"], 6933, atol=1) and lost.loc["shedding", "MWh"] < 1e-6
assert np.isclose(lost.loc["industry", "value_EUR"], 300 * lost.loc["industry", "MWh"])
lost.round(0)

**2. Scarcity rents**

Write `scarcity_rent(result, gen, mc)`: the margin (price − marginal cost) × output that a plant earns over the week. Compute it for the OCGT and the CCGT in the stressed week, and say how much of the OCGT's rent comes from the hours at 300 €/MWh.

In [ ]:
def scarcity_rent(result: dict, gen: str, mc: float) -> float:
    """Margin (EUR) a plant earns over the week: sum of (price - marginal cost) x output."""
    ...  # YOUR CODE HERE

In [ ]:
ocgt = scarcity_rent(stressed, "ocgt", 128.9)
ccgt = scarcity_rent(stressed, "ccgt", 89.1)
scarce = np.isclose(stressed["price"], 300.0)
ocgt_from_scarcity = float(((stressed["price"] - 128.9) * stressed["gen"]["ocgt"].to_numpy())[scarce].sum())
assert np.isclose(ocgt_from_scarcity, 171.1 * 400 * scarce.sum(), rtol=1e-6), "full output in every scarcity hour"
assert np.isclose(ocgt, ocgt_from_scarcity), "the OCGT earns nothing outside the scarcity hours"
print(f"OCGT rent {ocgt:,.0f} EUR (all of it in {scarce.sum()} scarcity hours); CCGT rent {ccgt:,.0f} EUR")

### Core

**3. Who sets the scarcity price?**

Vary industry's willingness to pay over [150, 300, 600, 1,000] €/MWh in the stressed week (commercial stays at 1,000, VOLL at 3,000). For each, report the cost, the highest price, the number of hours at that price and the energy industry gives up. Check one case against PyPSA.

In [ ]:
def wtp_sweep(values) -> pd.DataFrame:
    """Stressed-week results for each industrial willingness to pay (EUR/MWh)."""
    ...  # YOUR CODE HERE

In [ ]:
sweep = wtp_sweep([150.0, 300.0, 600.0, 1000.0])
dr600 = DR.copy()
dr600.loc["industry", "wtp"] = 600.0
assert np.isclose(pypsa_week(stress=True, dr=dr600).objective, sweep.loc[600.0, "cost"], rtol=1e-9)
assert np.allclose(sweep.max_price, sweep.index), "the scarcity price is industry's willingness to pay"
assert sweep.cost.is_monotonic_increasing
sweep.round(1)

**4. Beyond demand response: shedding**

Make the stressed week harsher step by step: demand 5 %, 10 % and 15 % higher (`load_scale`). For each, report how much each consumer group gives up, how much is shed involuntarily, the highest price and the number of hours at VOLL. Check the harshest case against PyPSA.

In [ ]:
def harsher_weeks(scales) -> pd.DataFrame:
    """Per load scale: energy given up per group (MWh), shedding (MWh), highest price, hours at VOLL, cost."""
    ...  # YOUR CODE HERE

In [ ]:
harsh = harsher_weeks([1.05, 1.10, 1.15])
assert np.isclose(pypsa_week(stress=True, load_scale=1.15).objective, harsh.loc[1.15, "cost"], rtol=1e-9)
assert harsh.loc[1.05, "commercial"] > 0 and harsh.loc[1.05, "shed_MWh"] < 1e-6, "commercial first, no shedding yet"
assert harsh.loc[1.15, "shed_MWh"] > 0 and harsh.loc[1.15, "hours_at_voll"] > 0, "finally, involuntary shedding"
assert harsh.industry.is_monotonic_increasing and harsh.commercial.is_monotonic_increasing
harsh.round(0)

**5. Interpretation**

In task 3 the scarcity price simply follows industry's willingness to pay. What does that imply for how prices in tight hours are set in a market with flexible demand — and for a regulator who wants to cap prices? Use task 4 to say what happens when the flexible consumers are exhausted.

*Your answer here.*

### Challenge

**6. Shifting versus pumping, at the same size**

Lesson 7's pumped storage (300 MW, 2,400 MWh, 75 % round trip) saved 405,156 € in the normal week. Give load shifting the same size — 300 MW and a 2,400 MWh backlog — and compare. Then shrink the backlog over [300, 600, 1,200, 2,400] MWh: at which size does the extra backlog stop being worth much?

In [ ]:
def shifting_value(p: float, backlogs) -> pd.DataFrame:
    """Saving (EUR) against no shifting, normal week, for each backlog limit (MWh) at power p (MW)."""
    ...  # YOUR CODE HERE

In [ ]:
value = shifting_value(300.0, [300.0, 600.0, 1200.0, 2400.0])
PUMPED_SAVING = 405_156                                          # lesson 7, same week, same power and energy
assert value.loc[2400.0, "saving"] > PUMPED_SAVING, "no losses: the same size is worth more"
assert value.saving.is_monotonic_increasing
print(f"shifting 300 MW / 2,400 MWh saves {value.loc[2400.0, 'saving']:,.0f} EUR vs pumped storage {PUMPED_SAVING:,} EUR")
value.round(1)

**7. Interpretation**

Why is lossless shifting worth more than pumped storage of the same size, and where does the marginal value of extra backlog fall away? Name what the model leaves out that would make real load shifting worth less than this.

*Your answer here.*